Tomasz Lepszy, *475898*

# Meeting 3 — Generation from training photos

**Where things stand**

| Step | Status | Where |
| --- | --- | --- |
| Generate 8 pools (img2img × 4 strengths, IP-Adapter × 4 scales) on a Colab T4 | ⬜ to do | [colab/m3_train.ipynb](../colab/m3_train.ipynb) |
| Train 8 runs × 5 seeds | ⬜ to do | [colab/m3_train.ipynb](../colab/m3_train.ipynb) |
| Contact sheets: near copies vs new poses and backgrounds | ⬜ to do | section 2 |
| Validation analysis and sweep plots | ⬜ to do | sections 4, 6.1 |
| Pick the best run of each mode | ⬜ to do | sections 5.1, 6.1 |
| Test the two picked runs once | ⬜ to do | [colab/m3_test.ipynb](../colab/m3_test.ipynb), section 5.2 |
| Test interpretation | ⬜ to do | section 6.2 |

**Contents**
1. Problem: where Meetings 1 and 2 left off, the question, the expectations
2. The pools: settings, generation log, contact sheets, same photo across settings
3. What was done
4. Validation results: both sweeps, the other breeds, the hard Maus
5. Picking the run of each mode to test, and the test results
6. Interpretation: how close to the real photos, conclusion, limitations

This notebook only **reads and analyses**. Generation, training and testing need a GPU and live in `colab/` (shared code: `colab/colab_tools.py`). Run it locally after `git pull`; cells whose results don't exist yet print "Skipped". The generated images are on Google Drive (`MyDrive/dane_syntetyczne/pools/`), not in git: everything here is drawn from the manifests, `pool.json` files, contact sheets and `runs/`.

### Setup
Moves to the repository root, makes `synthdata` importable, checks that Oxford-IIIT Pet is in `./data`, and lists which Meeting 3 pools and runs exist yet.

In [ ]:
import csv
import itertools
import json
import os
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

here = Path.cwd()
os.chdir(next(p for p in [here, *here.parents] if (p / "pyproject.toml").exists()))
if str(Path("src").resolve()) not in sys.path:
    sys.path.insert(0, str(Path("src").resolve()))

from synthdata import data
from synthdata.config import CLASSES, TARGET
from synthdata.inputs import source_photo

data.download()  # skipped when ./data is already extracted
split = pd.DataFrame([p.__dict__ for p in data.load_split()])
missing = [i for i in split["image"] if not (data.image_path(i).exists() and data.trimap_path(i).exists())]
assert not missing, f"{len(missing)} photos missing from ./data, e.g. {missing[0]}"

ROOT = Path(".")
RUNS_DIR = ROOT / "runs"
POOLS_DIR = ROOT / "pools"
EVIDENCE_DIR = ROOT / "evidence"
EVIDENCE_DIR.mkdir(exist_ok=True)

BASELINES = ["m1-real", "m1-oversample", "m1-augment-strong"]  # Meeting 1, all tested
M2_RUN = "m2-base"                                              # Meeting 2 pool taken forward, tested
STRENGTHS = [0.3, 0.5, 0.7, 0.9]
IP_SCALES = [0.3, 0.5, 0.7, 1.0]
IMG2IMG_RUNS = [f"m3-strength-{s}" for s in STRENGTHS]
IP_RUNS = [f"m3-ip-{s}" for s in IP_SCALES]
POOL_RUNS = IMG2IMG_RUNS + IP_RUNS
REFERENCE_RUNS = BASELINES + [M2_RUN]
SETTING = {**{r: ("img2img", "strength", s) for r, s in zip(IMG2IMG_RUNS, STRENGTHS)},
           **{r: ("ip-adapter", "ip_adapter_scale", s) for r, s in zip(IP_RUNS, IP_SCALES)}}

COLORS = {
    "m1-real": "tab:gray", "m1-oversample": "tab:blue", "m1-augment-strong": "tab:red", "m2-base": "black",
    **{r: plt.get_cmap("Purples")(0.35 + 0.6 * i / 3) for i, r in enumerate(IMG2IMG_RUNS)},
    **{r: plt.get_cmap("Oranges")(0.35 + 0.6 * i / 3) for i, r in enumerate(IP_RUNS)},
}

# Which pools and runs exist yet (Colab pushes them one at a time)
HAVE_POOL = {r: (POOLS_DIR / r / "pool.json").exists() for r in POOL_RUNS}
HAVE_RUN = {r: (RUNS_DIR / r / "results.json").exists() for r in POOL_RUNS}
POOLS_DONE = [r for r in POOL_RUNS if HAVE_POOL[r]]
RUNS_DONE = [r for r in POOL_RUNS if HAVE_RUN[r]]
RUN_NAMES = REFERENCE_RUNS + RUNS_DONE
HAVE_RESULTS = bool(RUNS_DONE)
print(f"Ready: {Path.cwd()} | {len(split)} course photos")
print(f"Pools generated: {len(POOLS_DONE)}/8 | runs trained: {len(RUNS_DONE)}/8")
for r in POOL_RUNS:
    print(f"  {'✅' if HAVE_RUN[r] else ('🟡' if HAVE_POOL[r] else '⬜')} {r}")
if not HAVE_RESULTS:
    print("\nNo Meeting 3 runs yet: run colab/m3_train.ipynb in Colab, then `git pull` and re-run.")


def save(fig, name):
    """Save a figure to evidence/ and show it."""
    fig.savefig(EVIDENCE_DIR / name, dpi=160)
    plt.show()


LOOK = "background-color: #ffe08a; color: black"               # values the pick rests on
KEY = "background-color: #f4a259; color: black; font-weight: bold"  # the decisive value


def highlight(df, cells=(), key=(), precision=3):
    """Style a table: `cells` and `key` are lists of (row, column) to mark."""
    def styles(frame):
        out = pd.DataFrame("", index=frame.index, columns=frame.columns)
        for marks, style in [(cells, LOOK), (key, KEY)]:
            for row, col in marks:
                if row in out.index and col in out.columns:
                    out.loc[row, col] = style
        return out
    return df.style.apply(styles, axis=None).format(precision=precision)


def cells_of(rows, cols):
    return [(r, c) for r in rows for c in cols]

## 1. Problem

### 1.1 Where Meetings 1 and 2 left off

The course question is unchanged: *can images made by diffusion models improve recognition of Egyptian Mau, an under-represented cat breed, without harming the other seven breeds?* The bars so far (validation / **test**, mean over 5 seeds):

| Run | What the 55 extra Mau examples are | Mau recall val / **test** | Macro-F1 val / **test** |
| --- | --- | ---: | ---: |
| `m1-real` | none (15 Maus vs 70 per other breed) | 0.49 / **0.49** | 0.865 / **0.855** |
| `m1-oversample` | copies of the 15 photos | 0.69 / **0.70** | 0.894 / **0.882** |
| `m1-augment-strong` | strongly transformed copies | 0.78 / **0.73** | 0.903 / **0.885** |
| `m2-base` | SD 1.5 text-to-image, 4 prompts | 0.78 / **0.65** | 0.906 / **0.879** |

Meeting 2 showed that images drawn from a description alone often aren't Maus (tuxedo cats, striped tabbies, black cats for "bronze") and, on test, did worse than oversampling. This time the model also **sees the 15 training photos**, so the breed's coat should survive, and the open question becomes how far from those photos to go.

### 1.2 Question and expectations

> **How close to the 15 real training photos should the synthetic images be?** Near copies add little new information; images far from the photos may lose what makes the breed.

What I expected before generating (Meeting 0 and `EXPERIMENTS.md`):

1. **Low img2img strength (0.3) behaves like oversampling**: the image is almost the photo, so the result should be close to `m1-oversample`.
2. **Higher strength and IP-Adapter can add new poses and scenes**, which is what the hard validation Maus (#15, #155, #131) need.
3. **At strength 0.9 and IP scale 0.3 the photo has little influence**, so the pool should drift back towards Meeting 2's text-to-image failures.
4. **IP-Adapter keeps coat colour and pattern but draws a new scene**; img2img keeps the pose and background and changes more of the cat as strength grows.
5. **Bengal recall** is still the number to watch.
6. Things copied that shouldn't be (a photographer's signature, a studio backdrop) may be learned as "Mau" cues.

### 1.3 How I want to solve it

Two sweeps of four pools, one setting at a time. Every pool has 110 images, uses the four prompts and the negative prompt of `m2-base`, and cycles through the 15 training photos (image *i* starts from photo *i* mod 15 with seed 1000 + *i*, in every pool). Each training seed adds a random 55 of them, as in Meeting 2.

| Pool / run | Mode | Setting | Tags |
| --- | --- | ---: | --- |
| `m3-strength-0.3` | img2img | strength 0.3 (9 of 30 steps) | `mode=img2img`, `strength=0.3` |
| `m3-strength-0.5` | img2img | strength 0.5 (15 steps) | `mode=img2img`, `strength=0.5` |
| `m3-strength-0.7` | img2img | strength 0.7 (21 steps) | `mode=img2img`, `strength=0.7` |
| `m3-strength-0.9` | img2img | strength 0.9 (27 steps) | `mode=img2img`, `strength=0.9` |
| `m3-ip-0.3` | IP-Adapter | scale 0.3 | `mode=ip-adapter`, `ip_adapter_scale=0.3` |
| `m3-ip-0.5` | IP-Adapter | scale 0.5 | `mode=ip-adapter`, `ip_adapter_scale=0.5` |
| `m3-ip-0.7` | IP-Adapter | scale 0.7 | `mode=ip-adapter`, `ip_adapter_scale=0.7` |
| `m3-ip-1.0` | IP-Adapter | scale 1.0 | `mode=ip-adapter`, `ip_adapter_scale=1.0` |

Only the 15 **training** photos are used as sources (the tool refuses others), so the classifier is never evaluated on cats it has effectively seen. Everything else (split, EfficientNet-B0, 8 epochs, seeds, budget 55) is as in Meetings 1 and 2.

## 2. The pools

### 2.1 Settings, as recorded in each `pool.json`

In [ ]:
def pool_settings(name):
    return json.loads((POOLS_DIR / name / "pool.json").read_text())


if POOLS_DONE:
    settings_table = pd.DataFrame({
        name: {"mode": s["mode"], "strength": s["strength"], "ip_adapter_scale": s["ip_adapter_scale"],
               "adapter": s["adapter"], "model": s["model"], "steps": s["steps"], "guidance": s["guidance"],
               "prompts": len(s["prompts"]), "sources": len(s["sources"]), "seed of image 0": s["seed"]}
        for name in POOLS_DONE for s in [pool_settings(name)]
    }).T
    display(settings_table)
else:
    print("Skipped: no Meeting 3 pools yet.")

### 2.2 Generation log

Every attempt has a line in the pool's `manifest.jsonl`. img2img runs only steps × strength denoising steps (`effective_steps`).

In [ ]:
def manifest(name):
    return pd.DataFrame([json.loads(line) for line in (POOLS_DIR / name / "manifest.jsonl").read_text().splitlines() if line])


manifests = {name: manifest(name) for name in POOLS_DONE}
if manifests:
    display(pd.DataFrame({
        name: {"attempts": len(m), "ok": int((m["status"] == "ok").sum()),
               "safety blocked": int((m["status"] == "safety_blocked").sum()),
               "effective steps": int(m["effective_steps"].iloc[0]),
               "sources used": m["source"].nunique(),
               "s / image": round(m["seconds"].mean(), 1), "peak VRAM (MB)": int(m["peak_vram_mb"].max()),
               "GPU": m["gpu"].iloc[0]}
        for name, m in manifests.items()
    }).T)
else:
    print("Skipped: no Meeting 3 pools yet.")

### 2.3 Contact sheets

The first 24 images of each pool, each next to the training photo it started from (left: source, right: generated). Images #0–14 use the 15 training photos once each; #15–23 start again from the first nine, with other seeds and prompts.

In [ ]:
from IPython.display import Image as ShowImage, Markdown

for name in POOL_RUNS:
    sheet = POOLS_DIR / name / "contact-sheet.jpg"
    mode, key, value = SETTING[name]
    display(Markdown(f"#### `{name}` ({mode}, {key.replace('_', ' ')} {value})"))
    if sheet.exists():
        display(ShowImage(filename=str(sheet), width=1100))
    else:
        print("not generated yet")

### 2.4 The same training photo across settings

Cut from the contact sheets: for a few source photos, the photo itself followed by the image generated from it at each strength (top row) and each IP-Adapter scale (bottom row). Same photo, same seed, same prompt in every column, so the only change is the setting.

In [ ]:
TILE, CAPTION, COLUMNS = 192, 18, 4  # layout of synthdata.sheets.contact_sheet for image-conditioned pools
SHOWN_IMAGES = [0, 1, 2]  # image numbers (0-23) to show; edit after looking at the contact sheets


def generated_tile(name, index):
    """Generated image `index` (0-23) of a pool, cut from its contact sheet."""
    sheet = Image.open(POOLS_DIR / name / "contact-sheet.jpg")
    x, y = (index % COLUMNS) * 2 * TILE + TILE, (index // COLUMNS) * (TILE + CAPTION)
    return sheet.crop((x, y, x + TILE, y + TILE))


if len(POOLS_DONE) == len(POOL_RUNS):
    sweeps = [("strength", IMG2IMG_RUNS, STRENGTHS), ("IP scale", IP_RUNS, IP_SCALES)]
    fig, axes = plt.subplots(2 * len(SHOWN_IMAGES), 5, figsize=(11, 4.6 * len(SHOWN_IMAGES)))
    for i, index in enumerate(SHOWN_IMAGES):
        source = manifests[IMG2IMG_RUNS[0]]["source"].iloc[index]
        for j, (label, runs, values) in enumerate(sweeps):
            row = axes[2 * i + j]
            row[0].imshow(source_photo(source))
            row[0].set_title(f"#{index}: {source.replace('Egyptian_Mau_', 'photo ')}", fontsize=9)
            for ax, run, value in zip(row[1:], runs, values):
                ax.imshow(generated_tile(run, index))
                ax.set_title(f"{label} {value}", fontsize=9)
            for ax in row:
                ax.axis("off")
    fig.suptitle("Same training photo, same seed: img2img strength (upper rows) and IP-Adapter scale (lower rows)")
    fig.tight_layout()
    save(fig, "meeting3-same-source.png")
else:
    print(f"Skipped: {len(POOLS_DONE)}/8 pools generated.")

### 2.5 What the contact sheets show

*(Write after generation.)* For each mode: which images are near copies of the photo (same pose, background, framing), which add new poses or scenes and still look like a Mau (silver/bronze coat, random spots, M on the forehead, green eyes), and which lose the breed. Note anything copied that shouldn't be (a photographer's signature, a studio backdrop, a border).

| Observation | Pools | Examples (image #) |
| --- | --- | --- |
| near copy of the source photo | | |
| new pose / background, still a Mau | | |
| breed lost (tabby, tuxedo, Bengal-like) | | |
| artefact copied from the photo (signature, border) | | |

### 2.6 What each training seed actually got

Each seed draws its own 55 of the usable images. `run.json` lists them; the table counts how many of the 15 training photos each seed's images came from, so no seed is dominated by a few photos.

In [ ]:
rows = []
for name in RUNS_DONE:
    source_of = dict(zip(manifests[name]["sample_id"], manifests[name]["source"]))
    for k in range(5):
        run_json = RUNS_DIR / name / f"seed-{k}" / "run.json"
        if not run_json.exists():
            continue
        added = json.loads(run_json.read_text())["added"]
        counts = Counter(source_of[s] for s in added)
        rows.append({"run": name, "seed": k, "added": len(added), "photos used": len(counts),
                     "max per photo": max(counts.values()), "min per photo": min(counts.values())})
display(pd.DataFrame(rows).set_index(["run", "seed"]) if rows else "Skipped: no Meeting 3 runs yet.")

## 3. What was done

### 3.1 Generation and training
[colab/m3_train.ipynb](../colab/m3_train.ipynb) did everything on a Colab T4 GPU, one pool at a time:

```bash
synth-generate --name m3-strength-0.5 --device cuda --mode img2img --strength 0.5 \
    --prompt "<prompt 1>" ... --prompt "<prompt 4>" --negative-prompt "<negative>"
synth-generate --name m3-ip-0.5 --device cuda --mode ip-adapter --ip-adapter-scale 0.5 ...   # same prompts
synth-train --name m3-strength-0.5 --method synthetic --pool m3-strength-0.5 --device cuda \
    --tag mode=img2img --tag strength=0.5
synth-train --name m3-ip-0.5 --method synthetic --pool m3-ip-0.5 --device cuda \
    --tag mode=ip-adapter --tag ip_adapter_scale=0.5
```

After each pool it copied the images to Google Drive (and restores them from there on a new runtime), pushed the manifest, `pool.json` and contact sheet to GitHub, trained 5 seeds, copied the `model.pt` files to Drive and pushed the results.

### 3.2 Helpers
Same as Meeting 2: loading results, summaries, the paired sign-flip test, plots and the Mau error table.

In [ ]:
def load_runs(filename, run_names=None):
    """One row per (run, seed) from runs/<name>/<filename>."""
    records, missing = [], []
    for run_name in run_names or RUN_NAMES:
        path = RUNS_DIR / run_name / filename
        if not path.exists():
            missing.append(run_name)
            continue
        for seed in json.loads(path.read_text())["seeds"]:
            records.append({"run": run_name, "seed": seed["seed"],
                            "egyptian_mau_recall": seed["egyptian_mau_recall"],
                            "macro_f1": seed["macro_f1"], "accuracy": seed["accuracy"],
                            "bengal_recall": seed["recall"]["Bengal"]})
    if missing:
        print(f"No {filename} yet for:", ", ".join(missing))
    return pd.DataFrame(records)


def load_breed_recall(filename, run_names=None):
    """One row per (run, seed, breed) with that breed's recall."""
    rows = []
    for run_name in run_names or RUN_NAMES:
        for seed in json.loads((RUNS_DIR / run_name / filename).read_text())["seeds"]:
            for breed, value in seed["recall"].items():
                rows.append({"run": run_name, "seed": seed["seed"], "breed": breed, "recall": value})
    return pd.DataFrame(rows)


def summed_confusion(run_name, filename="metrics.json"):
    """Confusion matrix summed over the 5 seeds (rows = true, columns = predicted)."""
    total, names = None, None
    for k in range(5):
        m = json.loads((RUNS_DIR / run_name / f"seed-{k}" / filename).read_text())
        names = list(m["recall"])
        c = np.array(m["confusion"])
        total = c if total is None else total + c
    return total, names


def summarize(df, run_names=None):
    return (
        df.groupby("run", sort=False)
        .agg(seeds=("seed", "count"),
             recall_mean=("egyptian_mau_recall", "mean"), recall_std=("egyptian_mau_recall", "std"),
             recall_min=("egyptian_mau_recall", "min"), recall_max=("egyptian_mau_recall", "max"),
             macro_f1_mean=("macro_f1", "mean"), macro_f1_std=("macro_f1", "std"),
             bengal_recall=("bengal_recall", "mean"), accuracy_mean=("accuracy", "mean"))
        .reindex([r for r in run_names or RUN_NAMES if r in set(df["run"])])
        .round(3)
    )


def sign_flip_p(diffs):
    """Exact two-sided paired sign-flip (permutation) test over all 2^n sign patterns."""
    diffs = np.asarray(diffs, dtype=float)
    observed = abs(diffs.mean())
    signs = np.array(list(itertools.product([1, -1], repeat=len(diffs))))
    return float((np.abs((signs * diffs).mean(axis=1)) >= observed - 1e-12).mean())


def paired_tests(df, pairs, metrics=("egyptian_mau_recall", "macro_f1")):
    rows = []
    for metric in metrics:
        wide = df.pivot(index="seed", columns="run", values=metric)
        for a, b in pairs:
            d = wide[a] - wide[b]
            rows.append({"metric": metric, "comparison": f"{a} − {b}",
                         "mean diff": round(d.mean(), 3), "seeds better": int((d > 0).sum()),
                         "seeds equal": int((d == 0).sum()), "seeds worse": int((d < 0).sum()),
                         "p (sign-flip)": round(sign_flip_p(d), 3)})
    return pd.DataFrame(rows)


def plot_metric(df, metric, ylabel, title, run_names, ax, ylim=(0, 1.05)):
    """Dots = seeds, black bar = mean ± std."""
    for position, run_name in enumerate(run_names):
        values = df.loc[df["run"] == run_name, metric]
        ax.scatter([position] * len(values), values, alpha=0.8, color=COLORS[run_name],
                   label="individual seed" if position == 0 else None)
        ax.errorbar(position, values.mean(), yerr=values.std(ddof=1), fmt="_", color="black",
                    capsize=5, markersize=18, linewidth=2, label="mean ± std" if position == 0 else None)
    ax.set_xticks(range(len(run_names)), run_names, rotation=25, ha="right")
    ax.set_ylim(*ylim)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.25)
    ax.legend(loc="lower right")


def mau_error_table(run_names, filename="metrics.json"):
    """Mau row and column of the 5-seed summed confusion matrices."""
    rows = []
    for run_name in run_names:
        matrix, names = summed_confusion(run_name, filename)
        mau = names.index(TARGET)
        for j, name in enumerate(names):
            rows.append({"breed": name, "run": run_name,
                         "Mau predicted as": matrix[mau, j], "predicted as Mau": matrix[j, mau]})
    return pd.DataFrame(rows).pivot(index="breed", columns="run", values=["Mau predicted as", "predicted as Mau"]).reindex(
        columns=pd.MultiIndex.from_product([["Mau predicted as", "predicted as Mau"], run_names]))

## 4. Validation results

This section reports numbers; they are interpreted in section 6.1. 20 validation photos per breed, so one Mau photo = 0.05 recall. The Meeting 1 baselines and `m2-base` are shown alongside.

#### Per-seed results

In [ ]:
if HAVE_RESULTS:
    results = load_runs("results.json")
    display(results.pivot(index="seed", columns="run", values="egyptian_mau_recall")[RUN_NAMES].round(2))
    display(results.pivot(index="seed", columns="run", values="macro_f1")[RUN_NAMES].round(3))
else:
    print("Skipped: no Meeting 3 runs yet.")

#### Summary across seeds
`recall_vs_*` are the differences of the means from `m1-oversample`, `m1-augment-strong` and `m2-base`.

In [ ]:
if HAVE_RESULTS:
    summary = summarize(results)
    for ref, label in [("m1-oversample", "oversample"), ("m1-augment-strong", "strong"), ("m2-base", "m2")]:
        summary[f"recall_vs_{label}"] = (summary["recall_mean"] - summary.loc[ref, "recall_mean"]).round(3)
    summary.insert(0, "setting", [" ".join(map(str, SETTING[r][::2])) if r in SETTING else r[:2].upper() for r in summary.index])
    display(summary)
else:
    print("Skipped: no Meeting 3 runs yet.")

#### The two sweeps, setting on the x-axis

Each grey line is one training seed across the four settings (seed *k* of every run starts from the same weights, so the lines are the paired view); black is mean ± std. Dashed lines: `m1-oversample` (blue), `m1-augment-strong` (red), `m2-base` (black).

In [ ]:
def tag_value(run_name, key):
    """The numeric setting recorded by synth-train --tag in results.json."""
    return float(json.loads((RUNS_DIR / run_name / "results.json").read_text())["tags"][key])


if HAVE_RESULTS:
    METRICS = [("egyptian_mau_recall", "Egyptian Mau recall"), ("macro_f1", "Macro-F1"), ("bengal_recall", "Bengal recall")]
    sweeps = [("img2img strength", "strength", IMG2IMG_RUNS), ("IP-Adapter scale", "ip_adapter_scale", IP_RUNS)]
    fig, axes = plt.subplots(len(METRICS), 2, figsize=(13, 11), sharey="row")
    for col, (title, key, runs) in enumerate(sweeps):
        runs = [r for r in runs if r in RUNS_DONE]
        x = [tag_value(r, key) for r in runs]
        for row, (metric, label) in enumerate(METRICS):
            ax = axes[row, col]
            if runs:
                wide = results[results["run"].isin(runs)].pivot(index="seed", columns="run", values=metric)[runs]
                for _, values in wide.iterrows():
                    ax.plot(x, values, color="gray", alpha=0.35, lw=1)
                ax.errorbar(x, wide.mean(), yerr=wide.std(ddof=1), color="black", marker="o", capsize=5, lw=2)
            for ref in ["m1-oversample", "m1-augment-strong", "m2-base"]:
                ax.axhline(results.loc[results["run"] == ref, metric].mean(), color=COLORS[ref], ls="--", lw=1,
                           label=ref)
            ax.set_xticks(x if x else [])
            ax.grid(axis="y", alpha=0.25)
            if row == 0:
                ax.set_title(title)
            if row == len(METRICS) - 1:
                ax.set_xlabel(title)
            if col == 0:
                ax.set_ylabel(label)
                ax.legend(loc="lower left", fontsize=8)
    fig.suptitle("Meeting 3 validation: one setting at a time (black = mean ± std, grey = individual seeds)")
    fig.tight_layout()
    save(fig, "meeting3-validation-sweeps.png")
else:
    print("Skipped: no Meeting 3 runs yet.")

#### All runs side by side
Each dot is one seed; the black bar is mean ± 1 std over the 5 seeds.

In [ ]:
if HAVE_RESULTS:
    fig, axes = plt.subplots(1, 3, figsize=(20, 5.3))
    for ax, (metric, label, ylim) in zip(axes, [("egyptian_mau_recall", "Egyptian Mau recall", (0.35, 1.0)),
                                                ("macro_f1", "Macro-F1", (0.84, 0.94)),
                                                ("bengal_recall", "Bengal recall", (0.6, 1.0))]):
        plot_metric(results, metric, label, f"Validation: {label}", RUN_NAMES, ax, ylim)
        for ref in ["m1-oversample", "m1-augment-strong", "m2-base"]:
            ax.axhline(results.loc[results["run"] == ref, metric].mean(), color=COLORS[ref], ls="--", lw=1, alpha=0.7)
        ax.axvline(len(REFERENCE_RUNS) - 0.5, color="lightgray")
    fig.tight_layout()
    save(fig, "meeting3-validation-recall-f1.png")
else:
    print("Skipped: no Meeting 3 runs yet.")

#### Seed-by-seed differences and significance

With 5 seeds the smallest possible p is 2/32 = 0.0625. **Does each pool beat the bars?**

In [ ]:
if HAVE_RESULTS:
    display(paired_tests(results, [(r, ref) for ref in ["m1-oversample", "m1-augment-strong", "m2-base"] for r in RUNS_DONE],
                         metrics=("egyptian_mau_recall",)))
else:
    print("Skipped: no Meeting 3 runs yet.")

#### Recall of every breed

Do the generated Maus cost the other breeds anything? Mean over 5 seeds; Bengal is the one to watch.

In [ ]:
if HAVE_RESULTS:
    breed_recall = load_breed_recall("results.json")
    display(breed_recall.groupby(["breed", "run"])["recall"].mean().unstack()[RUN_NAMES].round(2))
else:
    print("Skipped: no Meeting 3 runs yet.")

#### Where the Egyptian Mau errors go

Summed over 5 seeds: what the 100 validation Mau predictions per run were classified as, and which other breeds were wrongly called Mau (out of 100 per breed).

In [ ]:
if HAVE_RESULTS:
    mau_errors = mau_error_table(RUN_NAMES)
    display(mau_errors["Mau predicted as"])
    display(mau_errors["predicted as Mau"])
else:
    print("Skipped: no Meeting 3 runs yet.")

#### Which validation Maus are missed

In how many of the 5 models per run each validation Mau was **not** recognised, for the photos missed at least once. Meeting 1's hard photos were #15, #155, #131 (never recognised), #152 and #171.

In [ ]:
if HAVE_RESULTS:
    rows = []
    for run_name in RUN_NAMES:
        for k in range(5):
            with open(RUNS_DIR / run_name / f"seed-{k}" / "predictions.csv") as stream:
                for r in csv.DictReader(stream):
                    if r["breed"] == TARGET:
                        rows.append({"photo": r["image"].replace("Egyptian_Mau_", "#"), "run": run_name, "predicted": r["predicted"]})
    mau_preds = pd.DataFrame(rows)
    wrong = mau_preds[mau_preds["predicted"] != TARGET]
    missed = wrong.groupby(["photo", "run"]).size().unstack(fill_value=0).reindex(columns=RUN_NAMES, fill_value=0)
    display(missed.sort_values(RUN_NAMES, ascending=False))
else:
    print("Skipped: no Meeting 3 runs yet.")

## 5. Picking the run of each mode to test, and the test results

### 5.1 Pick

**One** run of each mode is tested, chosen on validation only (reasoning in section 6.1). To test: set `TEST_IMG2IMG` and `TEST_IP` below, set the **same** values as `IMG2IMG_RUN` and `IP_RUN` in [colab/m3_test.ipynb](../colab/m3_test.ipynb), run that notebook in Colab, then `git pull` here and re-run from this cell. Once tested, don't change the choice.

In [ ]:
TEST_IMG2IMG = None  # ← e.g. "m3-strength-0.5": the img2img run kept after validation; never change after testing
TEST_IP = None       # ← e.g. "m3-ip-0.5": the IP-Adapter run kept after validation; never change after testing

PICKED = [r for r in (TEST_IMG2IMG, TEST_IP) if r]
assert TEST_IMG2IMG in IMG2IMG_RUNS + [None] and TEST_IP in IP_RUNS + [None]
TEST_RUNS = REFERENCE_RUNS + PICKED

if HAVE_RESULTS:
    pick_table = summarize(results, RUNS_DONE)[["recall_mean", "recall_std", "recall_min", "macro_f1_mean", "macro_f1_std", "bengal_recall"]]
    pick_table["Bengals called Mau"] = mau_errors["predicted as Mau"].loc["Bengal", RUNS_DONE]
    pick_table["picked"] = np.where(pick_table.index.isin(PICKED), "✅", "")
    display(highlight(pick_table, cells=cells_of(PICKED, pick_table.columns)))

# the Colab test notebook must test the same runs
colab_src = "".join("".join(c["source"]) for c in json.loads(Path("colab/m3_test.ipynb").read_text())["cells"])
for var, mine in [("IMG2IMG_RUN", TEST_IMG2IMG), ("IP_RUN", TEST_IP)]:
    colab_pick = next((r for r in POOL_RUNS if f'{var} = "{r}"' in colab_src), None)
    if colab_pick != mine:
        print(f"⚠ colab/m3_test.ipynb has {var} = {colab_pick!r}, this notebook {mine!r}: make them match.")

status = {n: (RUNS_DIR / n / "test-results.json").exists() for n in TEST_RUNS}
HAVE_TEST = len(PICKED) == 2 and all(status.values())
for n, done in status.items():
    print(f"{'✅' if done else '⬜'} {n}: {'tested' if done else 'no test results yet'}")
if not HAVE_TEST:
    print("\nPick both runs, run colab/m3_test.ipynb in Colab, then `git pull`. Section 5.2 is skipped until then.")

### 5.2 Test results

Evaluated once (240 test photos, 30 per breed, so one Mau photo = 0.033 recall), next to the Meeting 1 baselines and `m2-base`.

#### Per-seed results and summary

In [ ]:
if HAVE_TEST:
    test_results = load_runs("test-results.json", TEST_RUNS)
    display(test_results.pivot(index="seed", columns="run", values=["egyptian_mau_recall", "macro_f1"]).reindex(
        columns=pd.MultiIndex.from_product([["egyptian_mau_recall", "macro_f1"], TEST_RUNS])).round(3))
    test_summary = summarize(test_results, TEST_RUNS)
    display(test_summary)
else:
    print("Skipped: no test results yet.")

#### Validation vs test

In [ ]:
if HAVE_TEST:
    val_results = load_runs("results.json", TEST_RUNS)
    val_summary = summarize(val_results, TEST_RUNS)
    display(pd.DataFrame({
        "val recall": val_summary["recall_mean"], "test recall": test_summary["recall_mean"],
        "val macro-F1": val_summary["macro_f1_mean"], "test macro-F1": test_summary["macro_f1_mean"],
        "val Bengal": val_summary["bengal_recall"], "test Bengal": test_summary["bengal_recall"],
    }))

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
    for ax, (metric, label) in zip(axes, [("egyptian_mau_recall", "Egyptian Mau recall"), ("macro_f1", "Macro-F1")]):
        for offset, (name, df, marker) in zip([-0.12, 0.12], [("validation", val_results, "o"), ("test", test_results, "s")]):
            for position, run_name in enumerate(TEST_RUNS):
                values = df.loc[df["run"] == run_name, metric]
                ax.errorbar(position + offset, values.mean(), yerr=values.std(ddof=1), fmt=marker,
                            color=COLORS[run_name], mfc="white" if name == "test" else COLORS[run_name],
                            capsize=4, markersize=8, label=name if position == 0 else None)
        ax.set_xticks(range(len(TEST_RUNS)), TEST_RUNS, rotation=15)
        ax.set_ylabel(label)
        ax.set_title(f"{label}: validation (filled) vs test (open), mean ± std")
        ax.grid(axis="y", alpha=0.25)
        ax.legend()
    fig.tight_layout()
    save(fig, "meeting3-val-vs-test.png")
else:
    print("Skipped: no test results yet.")

#### Seed-by-seed significance and errors on test

In [ ]:
if HAVE_TEST:
    display(paired_tests(test_results, [(p, b) for p in PICKED for b in ["m1-oversample", "m1-augment-strong", "m2-base"]]))
    test_errors = mau_error_table(TEST_RUNS, "test-metrics.json")  # of 150 per breed
    display(test_errors)
else:
    print("Skipped: no test results yet.")

## 6. Interpretation

### 6.1 Validation (written before the test set is evaluated)

*(To write after training.)* The table of both sweeps; which strength and which scale are best on Mau recall without costing Bengal recall; whether the best of each mode beats `m1-oversample`, `m1-augment-strong` and `m2-base` on most seeds; and the pick of each mode, with the reason.

### 6.2 How close to the real photos should the synthetic images be?

*(To write after looking at the contact sheets and the sweeps.)* One paragraph, with image numbers from section 2.4: do near copies (strength 0.3, IP scale 1.0) help like oversampling or less; do new poses and backgrounds (strength 0.7–0.9, IP scale 0.3–0.5) help the hard Maus or lose the breed; where the sweet spot is.

### 6.3 Test

*(To write after colab/m3_test.ipynb.)* Test values of the two picked runs against Meetings 1 and 2, and whether the validation ranking held.

### 6.4 Conclusion

### 6.5 Limitations